In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import ray
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, Checkpoint

# ==========================================
# 1. PREPROCESSING & LABEL GENERATION
# ==========================================
def gaussian_label(length, center, sigma=10):
    x = np.arange(length)
    return np.exp(-0.5 * ((x - center) / sigma) ** 2)

def make_phase_labels(length, p_index=None, s_index=None, sigma=10):
    y = np.zeros((3, length), dtype=np.float32)

    if p_index is not None and 0 <= p_index < length:
        y[0] = gaussian_label(length, p_index, sigma)

    if s_index is not None and 0 <= s_index < length:
        y[1] = gaussian_label(length, s_index, sigma)

    phase_max = np.maximum(y[0], y[1])
    y[2] = 1.0 - phase_max

    y_sum = np.sum(y, axis=0, keepdims=True)
    y = y / np.maximum(y_sum, 1e-8)

    return y

def normalize_waveform(x):
    # Vectorized normalization across the batch
    means = np.mean(x, axis=2, keepdims=True)
    stds = np.std(x, axis=2, keepdims=True)
    return (x - means) / (stds + 1e-6)

def format_for_ml(batch):
    """
    Takes a Ray Dataset batch (dictionary of arrays) and converts it 
    into the 'x' and 'y' arrays expected by the PyTorch model.
    """
    batch_size = len(batch["p_arrival_sample"])
    seq_length = 6000
    
    # Initialize empty arrays for X (inputs) and Y (labels)
    x_batch = np.zeros((batch_size, 3, seq_length), dtype=np.float32)
    y_batch = np.zeros((batch_size, 3, seq_length), dtype=np.float32)
    
    for i in range(batch_size):
        # 1. Stack the 3 components into shape (3, 6000)
        x_batch[i, 0] = batch["denoised_waveform_N"][i]
        x_batch[i, 1] = batch["denoised_waveform_E"][i]
        x_batch[i, 2] = batch["denoised_waveform_Z"][i]
        
        # 2. Extract arrival samples safely (handling missing NaN values)
        p_val = batch["p_arrival_sample"][i]
        s_val = batch["s_arrival_sample"][i]
        
        p_idx = int(float(p_val)) if not np.isnan(float(p_val)) else None
        s_idx = int(float(s_val)) if not np.isnan(float(s_val)) else None
        
        # 3. Generate the Gaussian probability masks
        y_batch[i] = make_phase_labels(length=seq_length, p_index=p_idx, s_index=s_idx, sigma=10)
        
    # Apply normalization to the whole batch
    x_batch = normalize_waveform(x_batch)
    
    # Ray Train iter_torch_batches will automatically turn these into PyTorch Tensors
    return {"x": x_batch, "y": y_batch}


# ==========================================
# 2. PHASENET MODEL ARCHITECTURE
# ==========================================
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=7, padding=3),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=7, padding=3),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class PhaseNetLike(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, 32)
        self.enc2 = ConvBlock(32, 64)
        self.enc3 = ConvBlock(64, 128)
        self.enc4 = ConvBlock(128, 256)

        self.pool = nn.MaxPool1d(2)
        self.bottleneck = ConvBlock(256, 512)

        self.up4 = nn.ConvTranspose1d(512, 256, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(512, 256)

        self.up3 = nn.ConvTranspose1d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(256, 128)

        self.up2 = nn.ConvTranspose1d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(128, 64)

        self.up1 = nn.ConvTranspose1d(64, 32, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(64, 32)

        self.out = nn.Conv1d(32, out_channels, kernel_size=1)

    def crop_or_pad(self, x, target_length):
        current_length = x.shape[-1]
        if current_length == target_length:
            return x
        if current_length > target_length:
            return x[..., :target_length]
        pad_amount = target_length - current_length
        return F.pad(x, (0, pad_amount))

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.crop_or_pad(d4, e4.shape[-1])
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = self.crop_or_pad(d3, e3.shape[-1])
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = self.crop_or_pad(d2, e2.shape[-1])
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = self.crop_or_pad(d1, e1.shape[-1])
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        logits = self.out(d1)
        return F.softmax(logits, dim=1)


# ==========================================
# 3. PARALLEL RAY TRAINING INFRASTRUCTURE
# ==========================================
def soft_cross_entropy(pred, target, eps=1e-8):
    return -(target * torch.log(pred + eps)).sum(dim=1).mean()

def train_func(config):
    lr = config.get("lr", 1e-3)
    batch_size = config.get("batch_size", 16)
    epochs = config.get("epochs", 5)  

    model = PhaseNetLike()
    model = ray.train.torch.prepare_model(model)  
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_data_shard = ray.train.get_dataset_shard("train")

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        batch_count = 0

        # Efficiently stream batches out of Ray's distributed storage system
        train_loader = train_data_shard.iter_torch_batches(batch_size=batch_size, local_shuffle_buffer_size=250)
        
        for batch in train_loader:
            x = batch["x"] # Shape: (Batch, 3, 6000)
            y = batch["y"] # Shape: (Batch, 3, 6000)

            pred = model(x)
            loss = soft_cross_entropy(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            batch_count += 1

        avg_loss = total_loss / max(batch_count, 1)

        # Handle logging and custom file saving on Rank 0
        if ray.train.get_context().get_world_rank() == 0:
            print(f"Epoch {epoch + 1}: loss = {avg_loss:.5f}")
            
            output_dir = os.environ.get('HOME') + "/epak/model_checkpoints"
            os.makedirs(output_dir, exist_ok=True)
            
            checkpoint_filepath = os.path.join(output_dir, f"phasenet_epoch_{epoch + 1}.pt")
            torch.save(model.module.state_dict(), checkpoint_filepath)
            
            ray.train.report(
                metrics={"loss": avg_loss},
                checkpoint=Checkpoint.from_directory(output_dir)
            )
        else:
            ray.train.report(metrics={"loss": avg_loss})


# ==========================================
# 4. EXECUTION & INFERENCE ENGINE
# ==========================================
def pick_arrivals(probabilities, sampling_rate, window_start_time=0.0):
    p_prob = probabilities[0]
    s_prob = probabilities[1]

    p_index = int(np.argmax(p_prob))
    s_index = int(np.argmax(s_prob))

    p_time = window_start_time + p_index / sampling_rate
    s_time = window_start_time + s_index / sampling_rate

    return p_time, s_time, p_index, s_index

if __name__ == "__main__":
    if not ray.is_initialized():
        ray.init(num_cpus=8, ignore_reinit_error=True)

    input_dir = os.environ.get('HOME') + "/epak/denoised_waveforms_parquets"
    print(f"Loading data from {input_dir}...")
    raw_ds = ray.data.read_parquet(input_dir).limit(80_000)

    # Map the columns into tensors
    ml_ds = raw_ds.map_batches(format_for_ml, batch_format="numpy")

    # Split into 80% train, 20% test
    train_ds, test_ds = ml_ds.train_test_split(test_size=0.2)

    scaling_config = ScalingConfig(num_workers=4, use_gpu=False)

    trainer = TorchTrainer(
        train_loop_per_worker=train_func,
        train_loop_config={"lr": 2e-3, "batch_size": 16, "epochs": 5},
        scaling_config=scaling_config,
        datasets={"train": train_ds} 
    )

    print("--- Starting Parallel Distributed Ray Training Loop ---")
    result = trainer.fit()
    print("--- Training Execution Finished successfully! ---")

    # --- Run Inference using a real sample from your Test Split ---
    print("\n--- Running Inference Setup ---")
    eval_model = PhaseNetLike()
    
    final_epoch = 5

    checkpoint_dir = os.environ.get('HOME') + "/epak/model_checkpoints"
    local_checkpoint_path = os.path.join(checkpoint_dir, f"phasenet_epoch_{final_epoch}.pt")
    
    # Safety Check
    if not os.path.exists(local_checkpoint_path):
        raise FileNotFoundError(f"Missing weights at '{local_checkpoint_path}'. Run training first.")
        
    eval_model.load_state_dict(torch.load(local_checkpoint_path))
    eval_model.eval()

    # --- MODIFIED: Added iter() wrapper to fix the Ray TypeError ---
    test_sample = next(iter(test_ds.iter_batches(batch_size=1, batch_format="numpy")))
    real_x = test_sample["x"]  # Shape: (1, 3, 6000)

    with torch.no_grad():
        x_tensor = torch.tensor(real_x, dtype=torch.float32)
        pred_out = eval_model(x_tensor)[0].cpu().numpy() 

    p_time, s_time, p_idx, s_idx = pick_arrivals(pred_out, sampling_rate=100.0)

    print(f"Predicted P Wave Arrival Time: {p_time:.2f}s (Index: {p_idx})")
    print(f"Predicted S Wave Arrival Time: {s_time:.2f}s (Index: {s_idx})")

    # ==========================================
    # 5. FULL TEST DATASET EVALUATION
    # ==========================================
    print("\n--- Running Full Test Dataset Evaluation ---")
    
    total_test_loss = 0.0
    test_batch_count = 0

    test_loader = test_ds.iter_batches(batch_size=64, batch_format="numpy")

    with torch.no_grad():
        for batch in test_loader:
            x_tensor = torch.tensor(batch["x"], dtype=torch.float32)
            y_tensor = torch.tensor(batch["y"], dtype=torch.float32)

            pred = eval_model(x_tensor)
            loss = soft_cross_entropy(pred, y_tensor)
            
            total_test_loss += loss.item()
            test_batch_count += 1

    avg_test_loss = total_test_loss / max(test_batch_count, 1)
    final_train_loss = result.metrics["loss"]
    print("--------------------------------------------------")
    print(f"Final Training Loss: {final_train_loss:.5f}")
    print(f"Final Average Test Loss: {avg_test_loss:.5f}")
    print(f"Evaluated on {test_batch_count * 64} total samples.")
    print("--------------------------------------------------")

2026-05-30 14:56:04,650	INFO worker.py:1843 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


Loading data from /home/epak/epak/denoised_waveforms_parquets...


Metadata Fetch Progress 0:   0%|          | 0.00/104 [00:00<?, ? task/s]

Parquet Files Sample 0:   0%|          | 0.00/6.00 [00:00<?, ? file/s]

2026-05-30 14:56:43,062	INFO streaming_executor.py:108 -- Starting execution of Dataset. Full logs are in /tmp/ray/ray/session_2026-05-30_14-56-00_688085_913662/logs/ray-data
2026-05-30 14:56:43,063	INFO streaming_executor.py:109 -- Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> LimitOperator[limit=80000] -> TaskPoolMapOperator[MapBatches(format_for_ml)] -> AggregateNumRows[AggregateNumRows]


...

(pid=992972) Running 0: 0.00 row [00:00, ? row/s]

(pid=992972) - split(4, equal=True) 1: 0.00 row [00:00, ? row/s]

== Status ==
Current time: 2026-05-30 21:19:26 (running for 06:22:02.55)
Using FIFO scheduling algorithm.
Logical resource usage: 5.0/8 CPUs, 0/0 GPUs
Result logdir: /tmp/ray/ray/session_2026-05-30_14-56-00_688085_913662/artifacts/2026-05-30_14-57-23/TorchTrainer_2026-05-30_14-57-23/driver_artifacts
Number of trials: 1/1 (1 RUNNING)


== Status ==
Current time: 2026-05-30 21:19:31 (running for 06:22:07.57)
Using FIFO scheduling algorithm.
Logical resource usage: 5.0/8 CPUs, 0/0 GPUs
Result logdir: /tmp/ray/ray/session_2026-05-30_14-56-00_688085_913662/artifacts/2026-05-30_14-57-23/TorchTrainer_2026-05-30_14-57-23/driver_artifacts
Number of trials: 1/1 (1 RUNNING)


== Status ==
Current time: 2026-05-30 21:19:36 (running for 06:22:12.59)
Using FIFO scheduling algorithm.
Logical resource usage: 5.0/8 CPUs, 0/0 GPUs
Result logdir: /tmp/ray/ray/session_2026-05-30_14-56-00_688085_913662/artifacts/2026-05-30_14-57-23/TorchTrainer_2026-05-30_14-57-23/driver_artifacts
Number of trials: 1/1 (1 

(RayTrainWorker pid=992586) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/home/epak/ray_results/TorchTrainer_2026-05-30_14-57-23/TorchTrainer_89ee6_00000_0_2026-05-30_14-57-23/checkpoint_000004)
2026-05-30 22:59:08,542	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/epak/ray_results/TorchTrainer_2026-05-30_14-57-23' in 0.0301s.
2026-05-30 22:59:08,545	INFO tune.py:1041 -- Total run time: 28905.03 seconds (28904.97 seconds for the tuning loop).


== Status ==
Current time: 2026-05-30 22:59:08 (running for 08:01:44.97)
Using FIFO scheduling algorithm.
Logical resource usage: 5.0/8 CPUs, 0/0 GPUs
Result logdir: /tmp/ray/ray/session_2026-05-30_14-56-00_688085_913662/artifacts/2026-05-30_14-57-23/TorchTrainer_2026-05-30_14-57-23/driver_artifacts
Number of trials: 1/1 (1 TERMINATED)


== Status ==
Current time: 2026-05-30 22:59:08 (running for 08:01:45.00)
Using FIFO scheduling algorithm.
Logical resource usage: 5.0/8 CPUs, 0/0 GPUs
Result logdir: /tmp/ray/ray/session_2026-05-30_14-56-00_688085_913662/artifacts/2026-05-30_14-57-23/TorchTrainer_2026-05-30_14-57-23/driver_artifacts
Number of trials: 1/1 (1 TERMINATED)


--- Training Execution Finished successfully! ---

--- Running Inference Setup ---
Predicted P Wave Arrival Time: 8.05s (Index: 805)
Predicted S Wave Arrival Time: 10.02s (Index: 1002)

--- Running Full Test Dataset Evaluation ---
--------------------------------------------------
Final Training Loss: 0.01585
Final Ave

## FP and FN

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import ray

# ==========================================
# 1. SETUP & PREPROCESSING FUNCTIONS
# ==========================================
TOLERANCE_SAMPLES = 10  # 0.1 seconds at 100 Hz
SAMPLING_RATE = 100.0

def gaussian_label(length, center, sigma=10):
    x = np.arange(length)
    return np.exp(-0.5 * ((x - center) / sigma) ** 2)

def make_phase_labels(length, p_index=None, s_index=None, sigma=10):
    y = np.zeros((3, length), dtype=np.float32)
    if p_index is not None and 0 <= p_index < length:
        y[0] = gaussian_label(length, p_index, sigma)
    if s_index is not None and 0 <= s_index < length:
        y[1] = gaussian_label(length, s_index, sigma)
    phase_max = np.maximum(y[0], y[1])
    y[2] = 1.0 - phase_max
    y_sum = np.sum(y, axis=0, keepdims=True)
    return y / np.maximum(y_sum, 1e-8)

def normalize_waveform(x):
    means = np.mean(x, axis=2, keepdims=True)
    stds = np.std(x, axis=2, keepdims=True)
    return (x - means) / (stds + 1e-6)

def format_for_ml(batch):
    batch_size = len(batch["p_arrival_sample"])
    seq_length = 6000
    x_batch = np.zeros((batch_size, 3, seq_length), dtype=np.float32)
    y_batch = np.zeros((batch_size, 3, seq_length), dtype=np.float32)
    
    for i in range(batch_size):
        x_batch[i, 0] = batch["denoised_waveform_N"][i]
        x_batch[i, 1] = batch["denoised_waveform_E"][i]
        x_batch[i, 2] = batch["denoised_waveform_Z"][i]
        
        p_val = batch["p_arrival_sample"][i]
        s_val = batch["s_arrival_sample"][i]
        p_idx = int(float(p_val)) if not np.isnan(float(p_val)) else None
        s_idx = int(float(s_val)) if not np.isnan(float(s_val)) else None
        
        y_batch[i] = make_phase_labels(length=seq_length, p_index=p_idx, s_index=s_idx, sigma=10)
        
    x_batch = normalize_waveform(x_batch)
    return {"x": x_batch, "y": y_batch}

# ==========================================
# 2. MODEL ARCHITECTURE
# ==========================================
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=7, padding=3),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=7, padding=3),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class PhaseNetLike(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, 32)
        self.enc2 = ConvBlock(32, 64)
        self.enc3 = ConvBlock(64, 128)
        self.enc4 = ConvBlock(128, 256)
        self.pool = nn.MaxPool1d(2)
        self.bottleneck = ConvBlock(256, 512)
        self.up4 = nn.ConvTranspose1d(512, 256, kernel_size=2, stride=2)
        self.dec4 = ConvBlock(512, 256)
        self.up3 = nn.ConvTranspose1d(256, 128, kernel_size=2, stride=2)
        self.dec3 = ConvBlock(256, 128)
        self.up2 = nn.ConvTranspose1d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(128, 64)
        self.up1 = nn.ConvTranspose1d(64, 32, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(64, 32)
        self.out = nn.Conv1d(32, out_channels, kernel_size=1)

    def crop_or_pad(self, x, target_length):
        current_length = x.shape[-1]
        if current_length == target_length: return x
        if current_length > target_length: return x[..., :target_length]
        return F.pad(x, (0, target_length - current_length))

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.crop_or_pad(self.up4(b), e4.shape[-1]), e4], dim=1))
        d3 = self.dec3(torch.cat([self.crop_or_pad(self.up3(d4), e3.shape[-1]), e3], dim=1))
        d2 = self.dec2(torch.cat([self.crop_or_pad(self.up2(d3), e2.shape[-1]), e2], dim=1))
        d1 = self.dec1(torch.cat([self.crop_or_pad(self.up1(d2), e1.shape[-1]), e1], dim=1))
        return F.softmax(self.out(d1), dim=1)

# ==========================================
# 3. LOAD DATA & SAVED MODEL
# ==========================================
if not ray.is_initialized():
    ray.init(num_cpus=8, ignore_reinit_error=True)

print("Loading Data...")
input_dir = os.environ.get('HOME') + "/epak/denoised_waveforms_parquets"
# Limit to 50k to match your memory footprint
raw_ds = ray.data.read_parquet(input_dir).limit(50_000) 
ml_ds = raw_ds.map_batches(format_for_ml, batch_format="numpy")
_, test_ds = ml_ds.train_test_split(test_size=0.2)

print("Loading Saved Model...")
eval_model = PhaseNetLike()
checkpoint_path = os.environ.get('HOME') + "/epak/model_checkpoints/phasenet_epoch_5.pt"
eval_model.load_state_dict(torch.load(checkpoint_path, map_location=torch.device('cpu')))
eval_model.eval() 

# ==========================================
# 4. RUN PHYSICAL ACCURACY EVALUATION
# ==========================================
p_true_positives = 0
s_true_positives = 0
total_samples = 0

p_residuals = []
s_residuals = []

print("\n--- Running Evaluation (This will take a few minutes) ---")
test_loader = test_ds.iter_batches(batch_size=64, batch_format="numpy")

with torch.no_grad():
    for batch in test_loader:
        x_tensor = torch.tensor(batch["x"], dtype=torch.float32)
        true_y = batch["y"] 
        
        # Get model predictions
        pred = eval_model(x_tensor).cpu().numpy()
        
        # Loop through the batch
        for i in range(len(pred)):
            true_p_idx = np.argmax(true_y[i, 0])
            true_s_idx = np.argmax(true_y[i, 1])
            
            pred_p_idx = np.argmax(pred[i, 0])
            pred_s_idx = np.argmax(pred[i, 1])
            
            p_error = abs(pred_p_idx - true_p_idx)
            s_error = abs(pred_s_idx - true_s_idx)
            
            p_residuals.append(p_error / SAMPLING_RATE)
            s_residuals.append(s_error / SAMPLING_RATE)
            
            # True Positives vs False Positives (Misses)
            if p_error <= TOLERANCE_SAMPLES:
                p_true_positives += 1
            if s_error <= TOLERANCE_SAMPLES:
                s_true_positives += 1
                
            total_samples += 1

# Calculate Final Metrics
p_accuracy = (p_true_positives / total_samples) * 100
s_accuracy = (s_true_positives / total_samples) * 100

avg_p_error = np.mean(p_residuals)
avg_s_error = np.mean(s_residuals)

print("\n==================================================")
print(f"📊 PHYSICAL ACCURACY REPORT (Tolerance: ±0.1s)")
print("==================================================")
print(f"Total Test Samples Evaluated: {total_samples}\n")

print(f"P-WAVE PERFORMANCE:")
print(f"  - True Positives (Hit):  {p_true_positives}")
print(f"  - False Positives (Miss): {total_samples - p_true_positives}")
print(f"  - Accuracy:              {p_accuracy:.2f}%")
print(f"  - Average Error:         {avg_p_error:.4f} seconds\n")

print(f"S-WAVE PERFORMANCE:")
print(f"  - True Positives (Hit):  {s_true_positives}")
print(f"  - False Positives (Miss): {total_samples - s_true_positives}")
print(f"  - Accuracy:              {s_accuracy:.2f}%")
print(f"  - Average Error:         {avg_s_error:.4f} seconds")
print("==================================================")